In [1]:
from transformers import pipeline

In [2]:
# Load a smaller text-generation model from Hugging Face
model_name = "google/flan-t5-xl"  # Lightweight model for efficiency
pipe = pipeline("text2text-generation", model=model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [3]:
# Sample math word problem
math_problem = "A farmer has 12 apples. He gives 4 apples to a friend and buys 6 more. How many apples does he have now?"

In [5]:
# Direct Prompting
direct_prompt = f"Solve this problem and give only the answer: {math_problem}"
direct_response = pipe(direct_prompt, max_length=50)[0]["generated_text"].strip()

In [6]:
# Chain-of-Thought Prompting
cot_prompt = f"Solve this problem step-by-step, explaining your reasoning: {math_problem}"
cot_response = pipe(cot_prompt, max_length=100)[0]["generated_text"].strip()

In [7]:
expected_answer = "14"

In [8]:
# Display results
print("Direct Response:", direct_response)
print("\nCoT Response:", cot_response)


Direct Response: 12 + 6 = 16

CoT Response: He has 12 - 4 = 8 apples left. He buys 8 + 6 = 14 apples. So he has 8 + 14 = 28 apples. The final answer: 28.


In [9]:
criteria = {
    "accuracy": "Is the final answer correct?",
    "coherence": "Does the response follow a logical progression?",
    "relevance": "Does the response stay focused on the problem?"
}

print("\nEvaluation Criteria:")
for key, value in criteria.items():
    print(f"- {key.capitalize()}: {value}")


Evaluation Criteria:
- Accuracy: Is the final answer correct?
- Coherence: Does the response follow a logical progression?
- Relevance: Does the response stay focused on the problem?


In [10]:
# Evaluation Function
def evaluate_response(response, expected_answer):
    """Evaluates response based on accuracy, coherence, and relevance."""
    criteria_scores = {"accuracy": 0, "coherence": 0, "relevance": 0}

    # Accuracy: Check if the correct numerical answer appears
    if expected_answer in response:
        criteria_scores["accuracy"] = 1

    # Coherence: Check for logical structure (look for step-by-step words)
    coherence_keywords = ["first", "then", "next", "finally"]
    if any(word in response.lower() for word in coherence_keywords):
        criteria_scores["coherence"] = 1

    # Relevance: Ensure response discusses apples (stays on topic)
    if "apple" in response.lower():
        criteria_scores["relevance"] = 1

    return criteria_scores

# Evaluate Direct Prompting
direct_eval = evaluate_response(direct_response, expected_answer)

# Evaluate Chain-of-Thought Prompting
cot_eval = evaluate_response(cot_response, expected_answer)

# Display Results
print("\n=== Direct Prompting ===")
print(f"Response: {direct_response}")
print(f"Evaluation: {direct_eval}")

print("\n=== Chain-of-Thought Prompting ===")
print(f"Response: {cot_response}")
print(f"Evaluation: {cot_eval}")

# Summary Comparison
print("\n=== Overall Comparison ===")
print(f"- Direct Prompting Scores: {direct_eval}")
print(f"- Chain-of-Thought Scores: {cot_eval}")

# Determine the better approach
direct_score = sum(direct_eval.values())
cot_score = sum(cot_eval.values())

if cot_score > direct_score:
    print("\n✅ Chain-of-Thought (CoT) prompting provides better response quality!")
elif cot_score < direct_score:
    print("\n🔴 Direct prompting performed better (unexpected).")
else:
    print("\n⚖️ Both performed equally.")



=== Direct Prompting ===
Response: 12 + 6 = 16
Evaluation: {'accuracy': 0, 'coherence': 0, 'relevance': 0}

=== Chain-of-Thought Prompting ===
Response: He has 12 - 4 = 8 apples left. He buys 8 + 6 = 14 apples. So he has 8 + 14 = 28 apples. The final answer: 28.
Evaluation: {'accuracy': 1, 'coherence': 0, 'relevance': 1}

=== Overall Comparison ===
- Direct Prompting Scores: {'accuracy': 0, 'coherence': 0, 'relevance': 0}
- Chain-of-Thought Scores: {'accuracy': 1, 'coherence': 0, 'relevance': 1}

✅ Chain-of-Thought (CoT) prompting provides better response quality!
